# CK-M worked solution: part B, the defect and the fix

*Energy Systems, Step by Step* · Modeler checkpoint CK-M. Run the supplied model as delivered, check it, find the defect, fix it and solve again (tasks 6 to 8).

In [ ]:
import json, os
import numpy as np
import scipy
from scipy.optimize import linprog

print("NumPy", np.__version__, "· SciPy", scipy.__version__)
major, minor = (int(v) for v in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this notebook needs SciPy 1.9 or later (pip install -U scipy)"

In [ ]:
# The supplied model's three representative days of 2025 (real SPP shapes; see the checkpoint page)
demand = {
    "winter": [277.65, 275.85, 275.85, 277.2, 281.7, 292.5, 310.05, 319.95, 315.45, 302.85, 292.05, 283.95, 275.85, 269.1, 264.15, 261, 263.25, 274.05, 284.85, 287.55, 288, 284.85, 277.2, 270],
    "shoulder": [231.3, 226.8, 223.65, 222.75, 224.55, 231.3, 244.8, 255.15, 257.85, 259.65, 261.45, 262.8, 261.9, 262.8, 262.8, 262.35, 263.7, 264.6, 263.25, 261, 263.25, 256.5, 244.8, 232.65],
    "summer": [257.85, 248.85, 242.55, 238.95, 239.85, 246.6, 261, 269.55, 278.1, 290.7, 308.7, 328.95, 348.75, 365.85, 377.1, 383.4, 386.55, 383.4, 373.5, 359.55, 347.85, 329.85, 308.7, 288.45],
}   # MW
avail = {
    "wind": {
        "winter": [0.4741, 0.4722, 0.4645, 0.4677, 0.4645, 0.4459, 0.4248, 0.3646, 0.289, 0.1768, 0.0942, 0.075, 0.0871, 0.1205, 0.1768, 0.2345, 0.3024, 0.3851, 0.469, 0.5049, 0.5279, 0.5324, 0.5331, 0.5209],
        "shoulder": [0.4421, 0.4235, 0.4145, 0.4062, 0.3921, 0.3652, 0.3466, 0.3332, 0.3678, 0.4158, 0.4414, 0.4286, 0.3998, 0.3806, 0.3614, 0.3441, 0.3453, 0.3139, 0.2774, 0.2102, 0.2031, 0.2236, 0.2383, 0.2665],
        "summer": [0.3562, 0.3364, 0.3312, 0.3389, 0.3319, 0.3261, 0.3184, 0.2966, 0.2351, 0.1403, 0.1211, 0.1307, 0.1378, 0.1454, 0.1538, 0.1647, 0.1877, 0.2178, 0.2242, 0.2518, 0.3453, 0.4338, 0.4812, 0.4754],
    },
    "solar": {
        "winter": [0, 0, 0, 0, 0, 0, 0, 0.001, 0.1158, 0.3996, 0.4939, 0.4447, 0.3617, 0.3709, 0.3494, 0.4324, 0.3084, 0.0717, 0, 0, 0, 0, 0, 0],
        "shoulder": [0, 0, 0, 0, 0, 0, 0, 0.0102, 0.2203, 0.4488, 0.4509, 0.5472, 0.5226, 0.5769, 0.6292, 0.5697, 0.497, 0.4355, 0.333, 0.0615, 0, 0, 0, 0],
        "summer": [0, 0, 0, 0, 0, 0, 0, 0.0236, 0.2285, 0.6486, 0.8249, 0.8474, 0.8136, 0.8074, 0.788, 0.8331, 0.789, 0.787, 0.4478, 0.0728, 0, 0, 0, 0],
    },
}   # output per MW built, 0 to 1

In [ ]:
DAYS = {"winter": 121, "shoulder": 122, "summer": 122}       # days of 2025 each day stands for
weight = {d: DAYS[d] / 365 for d in DAYS}                    # share of the year
cost = {"wind": 0, "solar": 0, "cc": 40, "pk": 100}           # running cost, $ per MWh
capital = {"wind": 100_000, "solar": 70_000, "cc": 80_000, "pk": 50_000}   # $ per MW-year, annualized
V = 10_000                                                    # $ per MWh of demand left unserved
G = ["wind", "solar", "cc", "pk"]

def supplied_model(weight):
    # variables: K[g] (MW built), then p[g,d,t] (MW) and u[d,t] (MW) for every day and hour
    names = [f"K_{g}" for g in G] + [f"{v}_{d}_{t}" for d in DAYS for t in range(24) for v in [f"p_{g}" for g in G] + ["u"]]
    col = {n: j for j, n in enumerate(names)}
    c = np.zeros(len(names))
    for g in G:
        c[col[f"K_{g}"]] = capital[g]
    for d in DAYS:
        for t in range(24):
            for g in G:
                c[col[f"p_{g}_{d}_{t}"]] = weight[d] * cost[g]
            c[col[f"u_{d}_{t}"]] = weight[d] * V
    A_eq, b_eq, A_ub = [], [], []
    for d in DAYS:
        for t in range(24):
            r = np.zeros(len(names))                                   # balance, MW
            for g in G:
                r[col[f"p_{g}_{d}_{t}"]] = 1
            r[col[f"u_{d}_{t}"]] = 1
            A_eq.append(r); b_eq.append(demand[d][t])
            for g in G:                                                # output within what is built
                r = np.zeros(len(names)); r[col[f"p_{g}_{d}_{t}"]] = 1
                r[col[f"K_{g}"]] = -(avail[g][d][t] if g in avail else 1)
                A_ub.append(r)
    res = linprog(c, A_ub=np.array(A_ub), b_ub=np.zeros(len(A_ub)), A_eq=np.array(A_eq), b_eq=b_eq, bounds=(0, None), method="highs")
    assert res.status == 0, res.message
    x = dict(zip(names, res.x))
    return dict(K={g: x[f"K_{g}"] for g in G}, annual_cost=res.fun,
                demand_in_year=sum(weight[d] * demand[d][t] for d in DAYS for t in range(24)),
                unserved_in_year=sum(weight[d] * x[f"u_{d}_{t}"] for d in DAYS for t in range(24)))

def report(R):
    for g, label in [("wind", "Wind farm"), ("solar", "Solar farm"), ("cc", "Gas combined-cycle"), ("pk", "Gas peaker")]:
        print(f"Built: {label:20s} {R['K'][g]:8.1f} MW")
    print(f"Annual cost: ${R['annual_cost'] / 1e6:.1f} million")
    print(f"Demand in the year: {R['demand_in_year']:,.0f} MWh")
    print(f"Demand left unserved in the year: {R['unserved_in_year']:,.0f} MWh")

supplied = supplied_model(weight)
report(supplied)

## The checks

The report solves to optimal and every hour balances, but two checks fail:

- **The year's total against an independent reference.** The model's year holds about 6,757 MWh of demand. Prairie Town's 2025 demand is about 2,470 GWh (the course's year file), and the three days weighted by their days hold 2,466 GWh. The model's year is one average day.
- **Units of the objective.** Capital is in \$ a year; running costs, multiplied by shares of the year that sum to 1, are in \$ for one average day.

Both point at line 2: `weight = DAYS[d] / 365`. The paper says the weight is the number of days.

In [ ]:
print(f"Weights sum to {sum(weight.values()):.3f}; the paper says 365")
print(f"Demand in the model's year: {supplied['demand_in_year']:,.1f} MWh; the days weighted by their days hold {sum(DAYS[d] * demand[d][t] for d in DAYS for t in range(24)) / 1000:,.1f} GWh")
assert abs(supplied["K"]["pk"] - 308.7) < 0.01 and supplied["K"]["wind"] < 1e-6 and supplied["K"]["solar"] < 1e-6
assert abs(supplied["demand_in_year"] - 6756.9) < 0.1

## The fix, and the fixed plan

In [ ]:
fixed = supplied_model(dict(DAYS))       # weight = days
report(fixed)
print(f"Cost per MWh of demand: ${fixed['annual_cost'] / fixed['demand_in_year']:.2f}")
assert abs(fixed["annual_cost"] - 106817573.9) < 50
assert abs(fixed["K"]["wind"] - 548.4) < 0.1 and abs(fixed["K"]["solar"] - 210.7) < 0.1
assert abs(fixed["K"]["cc"] - 151.9) < 0.1 and abs(fixed["K"]["pk"] - 54.3) < 0.1
assert fixed["unserved_in_year"] < 1e-6

**Why the symptom looked as it did.** With shares, the running costs of one average day meet a whole year of capital costs, so fuel looks almost free: the cheapest capital wins (peakers), and leaving demand unserved looks cheap too. The fixed plan answers the model as written; its costs are illustrative, and three typical days keep no extreme day (M3 keeps the extremes).